# Validation-controlled MLP on unseen airfoils

Run this next experiment in a fresh **Colab CPU** runtime after this notebook and its source helpers are published. Notebook 04 remains the historical budget experiment: its 100-iteration control beat the variants that stopped at 107. This new method is implemented but has not yet run on the NASA dataset; it does not promise R-squared 0.99.

There is **no hard 107-epoch cap** here. Training-loss stopping is effectively disabled. One bootstrap epoch followed by public `partial_fit` epochs preserves Adam and RNG state; feature and target scalers fit only once on the full training partition. Changed sampling/`partial_fit` behavior makes the new control a new method, not a bitwise reproduction of the old 100-iteration model.

Validation selects checkpoints and variants. Test evaluation is disabled by default and belongs only in the final cell. Entire test airfoil IDs are already unseen during fitting; duplicate/near-duplicate shape auditing and external-airfoil evaluation remain separate work. No new targets are silently added.

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

PROJECT_ROOT = Path('/content/airfoil-ml-surrogate')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/gaurav-zz/ML-for-CFD.git', str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / '.git').exists():
    raise FileNotFoundError('Expected a Git checkout. Use a fresh Colab runtime.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip():
    raise RuntimeError('The checkout has local edits. Preserve them and use a fresh runtime.')
previous_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=PROJECT_ROOT, check=True)
SOURCE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
if SOURCE_REVISION != previous_revision and any(name == 'src' or name.startswith('src.') for name in sys.modules):
    raise RuntimeError('Source changed after import. Restart the session and run all cells again.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Source revision: {SOURCE_REVISION}')

## Restore the frozen baseline

Leave `BASELINE_RUN = None` to choose the latest saved baseline `run.json`, or specify the original baseline file. Do not rerun notebook 02 to obtain a new split. The loader verifies the processed CSV SHA256, feature order, and exact saved geometry partitions. Targets remain those recorded by the baseline, currently Cl and Cd; adding Cm requires a separate label-coverage and multi-target study.

To reopen, set `EXISTING_RUN` to a completed `experiments/mlp_validation/<timestamp>` Drive directory. With `BASELINE_RUN = None`, its recorded baseline is restored. Dataset, feature, split, target-order, and scikit-learn-version mismatches are rejected before any model loading. Reopening skips all fitting and reuses existing report files; interrupted runs are not automatically resumed. Keep `RUN_FINAL_TEST = False` during tuning.

In [ ]:
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/airfoil-ml-surrogate')
PROCESSED_PATH = DRIVE_ROOT / 'data' / 'processed' / 'airfoil_learning.csv'
BASELINE_RUN = None
EXISTING_RUN = None
RUN_FINAL_TEST = False

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version
import json
import pandas as pd
from IPython.display import Image
from src.training.comparison import (
    VALIDATION_MLP_CONFIGS, compare_models, evaluate_selected_models, load_baseline_split,
)
from src.training.validation_reports import write_validation_reports

previous_report = None
if EXISTING_RUN is not None:
    EXISTING_RUN = Path(EXISTING_RUN)
    previous_report = json.loads((EXISTING_RUN / 'run.json').read_text(encoding='utf-8'))
    if previous_report['run_metadata'].get('experiment_type') != 'mlp_validation':
        raise ValueError('EXISTING_RUN must be an MLP validation experiment.')
    if previous_report['status'] not in ('completed', 'completed_with_failures'):
        raise ValueError('EXISTING_RUN must be completed; interrupted fits are not automatically resumed.')
    if BASELINE_RUN is None:
        BASELINE_RUN = previous_report['run_metadata']['baseline_run']
if BASELINE_RUN is None:
    candidates = sorted((DRIVE_ROOT / 'experiments' / 'baseline').glob('*/run.json'))
    if not candidates:
        raise FileNotFoundError('No saved baseline run.json was found. Restore the original baseline to Drive.')
    BASELINE_RUN = candidates[-1]
BASELINE_RUN = Path(BASELINE_RUN)
print(f'Restoring frozen baseline: {BASELINE_RUN}', flush=True)
split, feature_columns, target_columns, baseline_metadata = load_baseline_split(PROCESSED_PATH, BASELINE_RUN)
if previous_report is not None:
    for key in ('dataset', 'feature_columns', 'split_geometries', 'split_rows'):
        if previous_report['run_metadata'][key] != baseline_metadata[key]:
            raise ValueError(f'Existing experiment does not match the baseline: {key}')
    if previous_report['target_names'] != target_columns:
        raise ValueError('Existing experiment target names/order do not match.')
    if previous_report['run_metadata']['packages']['scikit-learn'] != version('scikit-learn'):
        raise RuntimeError('Use the saved scikit-learn version before loading the existing models, then restart the session.')
X_train, y_train = split.train[feature_columns], split.train[target_columns]
X_validation, y_validation = split.validation[feature_columns], split.validation[target_columns]
for partition in ('train', 'validation', 'test'):
    rows = baseline_metadata['split_rows'][partition]
    geometries = len(baseline_metadata['split_geometries'][partition])
    print(f'{partition}: {rows:,} rows, {geometries:,} geometries (saved partition)')
print('No geometry ID crosses partitions. Only training and validation enter model selection.')

## Two variants, one checkpoint per variant

The sequential queue compares `mlp_val_control` (learning rate 0.001) and `mlp_val_low_lr` (0.0005), both with hidden layers `(64, 32)`, batch size 1024, and seed 42. The shared `validation_training` policy is `max_epochs=400`, `check_every=5`, `patience=10`, `min_epochs=100`, `min_delta=0.001`. Evaluate all validation rows at epoch 1, every 5 epochs, and the final epoch. Patience counts checks without at least 0.1% relative progress against its progress reference; it cannot stop training before 100 epochs. An honest validation stop can still occur below 400. For an explicit full-budget experiment, set each config's `validation_training['patience'] = None` before fitting a new run.

Each variant saves the **single best observed joint checkpoint**, minimizing `mean_target((validation_RMSE / training_target_scale) ** 2)`. Training-only target scales give a balanced dimensionless objective rather than letting Cl's units dominate Cd. This is not an independently chosen best epoch for each target. Any new minimum can update the checkpoint even without resetting patience. The saved `mlp.joblib` is the restored best checkpoint, not the last epoch. Afterward, `compare_models` selects variants separately per target using ordinary full-row validation RMSE in original units.

External `epochs_completed` is authoritative because scikit-learn's `n_iter_` can reset under `partial_fit`. During a fit, inspect its `validation_state.json` and `training_history.csv` for periodic progress; root `validation_history.csv` aggregates completed variants. Ordinary model failures are recorded and later variants continue.

In [ ]:
MODEL_CONFIGS = deepcopy(previous_report['model_configs'] if previous_report is not None else VALIDATION_MLP_CONFIGS)
display(pd.DataFrame(MODEL_CONFIGS))

In [ ]:
if EXISTING_RUN is not None:
    RUN_OUTPUT = EXISTING_RUN
    validation_results = pd.read_csv(RUN_OUTPUT / 'comparison.csv')
    print(f'Using saved experiment without retraining: {RUN_OUTPUT}')
else:
    run_started = datetime.now(timezone.utc)
    RUN_OUTPUT = DRIVE_ROOT / 'experiments' / 'mlp_validation' / run_started.strftime('%Y%m%dT%H%M%S%fZ')
    run_metadata = {
        'experiment_type': 'mlp_validation', 'created_at_utc': run_started.isoformat(),
        'git_commit': SOURCE_REVISION, 'python_version': sys.version,
        'packages': {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'joblib', 'matplotlib']},
        'baseline_run': str(BASELINE_RUN),
        'dataset': baseline_metadata['dataset'], 'feature_columns': feature_columns,
        'split_geometries': baseline_metadata['split_geometries'], 'split_rows': baseline_metadata['split_rows'],
    }
    validation_results = compare_models(
        X_train, y_train, X_validation, y_validation, target_columns, RUN_OUTPUT,
        model_configs=MODEL_CONFIGS, run_metadata=run_metadata,
        train_groups=split.train['geometry_id'], validation_groups=split.validation['geometry_id'],
    )
print(f'Artifacts: {RUN_OUTPUT}')

## Validation-only reports

Compare the full-row metrics, actual epochs, best epochs, stop reasons, and held-out validation curves, not training loss alone. `training_time_seconds` includes validation checks and checkpoint I/O, so it is not directly comparable to notebook 04's fit-only timing. Training loss includes regularization on scaled targets; it is not held-out accuracy. Equal-airfoil summaries and worst-airfoil errors complement row-weighted selection.

The source helper creates missing reports and preserves existing files. Metrics use all validation rows; prediction plots use at most 5,000 points. The notebook displays the persisted images without regenerating plots. Sync locally with `python -m src.utils.sync_results`: root `run.json`, `comparison.csv`, `validation_by_geometry.csv`, `validation_history.csv`, `validation_summary.json`, and the five displayed PNGs are report-only artifacts under `results/colab/mlp_validation/<timestamp>`. Models, per-model progress, and datasets stay in Drive.

In [ ]:
report = json.loads((RUN_OUTPUT / 'run.json').read_text(encoding='utf-8'))
if report['status'] not in ('completed', 'completed_with_failures') or validation_results.empty:
    raise RuntimeError(f'No completed validation comparison is available. Inspect {RUN_OUTPUT / "run.json"}.')
display(validation_results.sort_values(['target', 'rmse', 'model']).set_index(['target', 'model']))
display(pd.DataFrame([
    {'model': name, 'status': result['status'],
     'epochs_completed': result.get('training', {}).get('epochs_completed'),
     'best_epoch': result.get('training', {}).get('best_epoch'),
     'stop_reason': result.get('training', {}).get('stop_reason'),
     'best_validation_score': result.get('training', {}).get('best_validation_score'),
     'training_time_seconds': result.get('training_time_seconds'),
     'warnings': '; '.join(item['message'] for item in result['warnings']),
     'error': result.get('error', {}).get('message')}
    for name, result in report['results'].items()
]))
summary = write_validation_reports(RUN_OUTPUT, split.validation, feature_columns, target_columns)
display(pd.DataFrame(summary['per_geometry_summary']).set_index(['target', 'model']))
print('Validation-selected variants:', summary['selected_models'])
for filename in ('validation_rmse.png', 'training_loss.png', 'validation_curves.png', 'worst_airfoils.png', 'validation_predictions.png'):
    display(Image(filename=str(RUN_OUTPUT / filename)))
print(f'Validation-only reports ready for local sync: {RUN_OUTPUT}')

## Optional final test after decisions are frozen

Keep `RUN_FINAL_TEST = False` during development. In the live session, set it to true and execute only the final cell. After reconnecting, set `EXISTING_RUN` to the completed validation run and restore it without refitting. The helper loads the saved validation-selected checkpoints; test scores never choose epochs or rerank variants. The baseline loader restores the original CSV partitions, but test features and labels are passed to evaluation only inside this opt-in cell.

Final evaluation adds `test_metrics.csv` and `test_evaluation.json`. Local sync creates a separate `<timestamp>_with_test` snapshot, preserving the validation-only snapshot. These are held-out NASA airfoil IDs, not proof of arbitrary external-shape or experimental aerodynamic accuracy.

In [ ]:
if RUN_FINAL_TEST:
    final_results = evaluate_selected_models(
        RUN_OUTPUT, split.test[feature_columns], split.test[target_columns], target_names=target_columns,
    )
    display(final_results.set_index(['target', 'model']))
else:
    print('Test evaluation skipped. Validation-only reports are complete and can be synced locally.')